# SimVerse — Train a Robot Arm on Colab GPU

This notebook trains a Panda robot arm to pick up a red cube from a desk.
After training, download the model and run it locally with the 3D viewer.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup

In [ ]:
# Install system + Python dependencies
!apt-get install -y -qq libosmesa6-dev libgl1-mesa-glx
!pip install -q mujoco gymnasium stable-baselines3 pydantic pyyaml shimmy mediapy

# Clone the repo (skip if already cloned)
import os
if not os.path.exists("/content/simulation-universe/src/simverse"):
    !git clone https://github.com/AmirKakon/simulation-universe.git /content/simulation-universe

# Remove any broken pip install of simverse
!pip uninstall simverse -y 2>/dev/null

# Point Python directly at the source code
import sys
os.environ["MUJOCO_GL"] = "osmesa"
sys.path.insert(0, "/content/simulation-universe/src")
os.chdir("/content/simulation-universe")

# Verify
import simverse
print(f"SimVerse {simverse.__version__} ready!")

import torch
gpu = torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU only"
print(f"Device: {gpu}")

## 2. Preview the Environment

In [ ]:
import gymnasium as gym
import simverse.envs
import mediapy
import numpy as np

env = gym.make("SimVerse/DeskPickup-v0", render_mode="rgb_array")
obs, info = env.reset(seed=42)

# Render from 3 camera angles
frames = []
for cam in ["overhead", "side", "front"]:
    engine = env.unwrapped.engine
    frame = engine.render(width=480, height=360, camera_name=cam)
    frames.append(frame.rgb)

mediapy.show_images(frames, titles=["Overhead", "Side", "Front"], height=240)
env.close()

print(f"Observation keys: {list(obs.keys())}")
print(f"Action space: {env.action_space.shape} (7 joints + 1 gripper)")

## 3. Random Actions (Before Training)

Watch what the arm does with random actions — it flails around uselessly.

In [ ]:
env = gym.make("SimVerse/DeskPickup-v0", render_mode="rgb_array")
obs, info = env.reset(seed=42)

frames = []
for step in range(300):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    if step % 3 == 0:
        frame = env.render()
        if frame is not None:
            frames.append(frame)
    if terminated or truncated:
        obs, info = env.reset()

env.close()
mediapy.show_video(frames, fps=15, title="Random Actions (untrained)")

## 4. Train the Policy

Adjust `total_timesteps` based on how long you want to wait:
- `100_000` — ~3 min (learns to reach toward the cube)
- `500_000` — ~15 min (learns to reach and maybe grasp)
- `1_000_000` — ~30 min (best chance at full pick-up)

In [ ]:
from simverse.training import Trainer, TrainingConfig, Algorithm

config = TrainingConfig(
    env_id="SimVerse/DeskPickup-v0",
    algorithm=Algorithm.PPO,
    total_timesteps=500_000,
    learning_rate=3e-4,
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=42,
)

trainer = Trainer(config)
trainer.setup()
model_path = trainer.train()
trainer.cleanup()

print(f"\nModel saved to: {model_path}")

## 5. Watch the Trained Policy

In [ ]:
from stable_baselines3 import PPO

trained_model = PPO.load(str(model_path))
env = gym.make("SimVerse/DeskPickup-v0", render_mode="rgb_array")

frames = []
obs, info = env.reset(seed=42)
total_reward = 0
successes = 0
episodes = 0

for step in range(1000):
    action, _ = trained_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    if step % 2 == 0:
        frame = env.render()
        if frame is not None:
            frames.append(frame)

    if terminated or truncated:
        episodes += 1
        if info.get("success", False):
            successes += 1
        obs, info = env.reset()

env.close()

print(f"Episodes: {episodes}, Successes: {successes}, Total reward: {total_reward:.1f}")
mediapy.show_video(frames, fps=15, title="Trained Policy")

## 6. Evaluate Performance

In [ ]:
from simverse.training.evaluation import evaluate_policy

env = gym.make("SimVerse/DeskPickup-v0")
result = evaluate_policy(trained_model, env, n_episodes=20)
env.close()

print(f"Mean reward:  {result.mean_reward:.2f} +/- {result.std_reward:.2f}")
print(f"Mean length:  {result.mean_length:.0f} steps")
print(f"Success rate: {result.success_rate:.1%}")

## 7. Download the Trained Model

Download and run locally with the interactive 3D viewer:
```
cd simulation-universe
python scripts/watch_trained.py --model path/to/final_model.zip
```

In [ ]:
from google.colab import files

# SB3 saves models as .zip
model_zip = str(model_path) + ".zip"
if not os.path.exists(model_zip):
    model_zip = str(model_path)

print(f"Downloading: {model_zip}")
files.download(model_zip)